In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [4]:
df = pd.read_csv("../data/raw/creditcard.csv")
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (250744, 50)


,transaction_id,user_id,device_id,merchant_id,session_id,txn_time,txn_amount,txn_currency,txn_hour,txn_day_of_week,...,avg_txn_amount_90d,dormant_account_flag,new_payee_added_24h,email_change_7d,phone_change_7d,geo_distance_last_login,impossible_travel_flag,login_hour_deviation,session_duration_zscore,fraud_label
0,1,16795,8512,3450,93177,26-04-2024 9:41,75.985614,USD,9,4,...,14.933552,0,0,0,0,1.460731,0,4.129832,-1.463200,0
1,2,1860,4155,3552,461632,21-01-2024 6:24,22.339063,CAD,6,6,...,9.163966,0,0,0,0,81.059175,0,0.534457,-0.823666,0
2,3,39158,12806,3480,261998,05-01-2024 13:16,113.278392,USD,13,4,...,229.594905,0,0,0,0,65.769144,0,-0.484853,0.443232,0
3,4,12284,3770,3273,266224,14-05-2024 23:53,45.821144,CAD,23,1,...,237.550036,0,0,0,0,44.559035,0,-1.228798,2.307250,0
4,5,7265,11772,3353,419057,20-02-2024 1:37,120.520888,CAD,1,1,...,15.370694,0,1,0,0,85.293193,0,3.144927,-1.884058,0


In [7]:
y = df['fraud_label']
X = df.drop(columns=['fraud_label'])


In [9]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Numeric features: 46
Categorical features: 3


C:\Users\harvi\AppData\Local\Temp\ipykernel_23352\3977206995.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [11]:
numeric_transformer = Pipeline(steps=[
   ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
   ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
   transformers=[
       ('num', numeric_transformer, numeric_features),
       ('cat', categorical_transformer, categorical_features)
   ]
)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
   X, y,
   test_size=0.2,
   stratify=y,
   random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (200595, 49)
Test shape: (50149, 49)


In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)


Processed train shape: (200595, 139825)


In [17]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']